# Chapter 09 — One Runtime, Many Windows

**Companion to *Applied AI*.**

This notebook accompanies Chapter 9. The chapter's architectural claim is that
if your state lives in the interface, you have as many processes as you have
interfaces. Its subtler claim is about what a capable model does with the gap.

## Question

**Can a second process continue work it never saw, using only an identifier —
and what happens to the work that a summary leaves out?**

## What this notebook establishes

- A durable append-only ledger in which three separate connections — standing
  in for three surfaces — create, continue and read back the same work by
  identity, with no shared variables between them.
- A context package: offered candidates, required items, a budget, a selection,
  and **a recorded reason for every exclusion**.
- The chapter's harder point, made executable: a *summary* of the same work
  silently drops the rejected alternative, and a component working from
  convention then proposes exactly the approach the project ruled out.

## What this notebook does **not** establish

- The ledger here is a **30-line teaching implementation**, not CodeAI's. The
  real one is inspected in the Chapter 14–18 notebooks.
- The "model" is a lookup table standing in for a learned prior. It is **not a
  language model**, and it demonstrates the *shape* of the failure, not its
  frequency.
- Nothing here measures multi-device synchronisation, offline operation, or
  concurrent writers. The chapter is explicit that CodeAI does not build them.

## Setup

A real file-backed SQLite database in a temporary directory, so that "a new
process" means something.

In [1]:
import json
import sqlite3
import tempfile
import uuid
from pathlib import Path

WORKDIR = Path(tempfile.mkdtemp(prefix="ch09-"))
LEDGER = WORKDIR / "ledger.sqlite"
print("ledger:", LEDGER)

def connect():
    """Every call is a fresh connection: nothing is shared but the file."""
    return sqlite3.connect(LEDGER)

def init():
    with connect() as c:
        c.execute("""CREATE TABLE IF NOT EXISTS events (
            seq INTEGER PRIMARY KEY AUTOINCREMENT,
            stream_id TEXT NOT NULL,
            kind TEXT NOT NULL,
            payload TEXT NOT NULL)""")

def append(stream_id, kind, payload):
    with connect() as c:
        c.execute("INSERT INTO events(stream_id, kind, payload) VALUES (?,?,?)",
                  (stream_id, kind, json.dumps(payload, sort_keys=True)))

def read(stream_id):
    with connect() as c:
        rows = c.execute(
            "SELECT seq, kind, payload FROM events WHERE stream_id=? ORDER BY seq",
            (stream_id,)).fetchall()
    return [(s, k, json.loads(p)) for s, k, p in rows]

init()
print("append-only: there is no UPDATE and no DELETE in this API")

ledger: $TMP\ch09-xwq599uk\ledger.sqlite
append-only: there is no UPDATE and no DELETE in this API


## Surface A — the laptop, Monday evening

Forty minutes of work. Three decisions settled, and **one approach explicitly
ruled out**. That last one is the interesting record.

In [2]:
TASK = "task-" + uuid.uuid4().hex[:8]

append(TASK, "task.created", {
    "objective": "Decide the cache TTL policy for the pricing API",
    "success": "a TTL is chosen, with the condition that would change it",
})
append(TASK, "decision.recorded", {
    "what": "TTL is chosen per environment, not globally",
    "why": "staging and production have different tolerance for staleness",
})
append(TASK, "decision.recorded", {
    "what": "TTL values live in config, not in code",
    "why": "operators change them without a deploy",
})
append(TASK, "approach.rejected", {
    "what": "a shared in-process cache keyed by request path",
    "why": "we ran this in 2025; it produced stale prices across tenants "
           "and the incident took two days to trace",
})

print("task id:", TASK)
print("(Surface A now exits. Nothing is held in memory.)")
for seq, kind, _ in read(TASK):
    print(f"  {seq}. {kind}")

task id: task-ef248d0f
(Surface A now exits. Nothing is held in memory.)
  1. task.created
  2. decision.recorded
  3. decision.recorded
  4. approach.rejected


## Surface B — the train, Tuesday morning

A different connection. It is given **only the task id**. It does not
reconstruct the work from a transcript; it opens it.

In [3]:
def open_work(task_id):
    """What a surface does instead of summarising a conversation."""
    events = read(task_id)
    if not events:
        raise KeyError(task_id)
    state = {"objective": None, "success": None,
             "decisions": [], "rejected": [], "calls": []}
    for _, kind, p in events:
        if kind == "task.created":
            state["objective"], state["success"] = p["objective"], p["success"]
        elif kind == "decision.recorded":
            state["decisions"].append(p)
        elif kind == "approach.rejected":
            state["rejected"].append(p)
        elif kind == "call.completed":
            state["calls"].append(p)
    return state

work = open_work(TASK)
print("objective :", work["objective"])
print("success   :", work["success"])
print("decisions :")
for d in work["decisions"]:
    print("   -", d["what"])
print("rejected  :")
for r in work["rejected"]:
    print("   -", r["what"])
    print("     because:", r["why"])

objective : Decide the cache TTL policy for the pricing API
success   : a TTL is chosen, with the condition that would change it
decisions :
   - TTL is chosen per environment, not globally
   - TTL values live in config, not in code
rejected  :
   - a shared in-process cache keyed by request path
     because: we ran this in 2025; it produced stale prices across tenants and the incident took two days to trace


## Context is selected, not forgotten

Surface B now compiles a context package for one call. The chapter's
distinction:

> **Memory is durable. Context is selected.**

Nothing recorded is lost. Items are excluded **deliberately**, and each
exclusion carries a reason.

In [4]:
def compile_context(task_id, budget):
    events = read(task_id)
    # Offer everything, in a canonical order, with a declared size.
    candidates = []
    for seq, kind, p in events:
        required = kind in ("task.created", "approach.rejected")
        size = len(json.dumps(p)) // 4          # crude token estimate
        candidates.append({"id": f"e{seq}", "kind": kind,
                           "required": required, "size": size, "payload": p})

    selected, trace, used = [], [], 0
    for c in sorted(candidates, key=lambda x: (not x["required"], x["id"])):
        if c["required"]:
            selected.append(c); used += c["size"]
            trace.append((c["id"], c["kind"], "included because required"))
        elif used + c["size"] <= budget:
            selected.append(c); used += c["size"]
            trace.append((c["id"], c["kind"], "included within budget"))
        else:
            trace.append((c["id"], c["kind"], "excluded because budget"))
    return selected, trace, used, len(candidates)

selected, trace, used, offered = compile_context(TASK, budget=40)

print(f"offered {offered} candidates, selected {len(selected)}, {used} of 40 tokens\n")
for cid, kind, reason in trace:
    mark = " " if reason.startswith("included") else "x"
    print(f"  [{mark}] {cid:<4} {kind:<20} {reason}")

print()
print("Every exclusion has a recorded reason. That is the difference between")
print("a system you can debug and one you can only apologise for.")

offered 4 candidates, selected 2, 75 of 40 tokens

  [ ] e1   task.created         included because required
  [ ] e4   approach.rejected    included because required
  [x] e2   decision.recorded    excluded because budget
  [x] e3   decision.recorded    excluded because budget

Every exclusion has a recorded reason. That is the difference between
a system you can debug and one you can only apologise for.


In [5]:
append(TASK, "call.completed", {
    "package": [c["id"] for c in selected],
    "excluded": [(cid, r) for cid, _, r in trace if r.startswith("excluded")],
    "answer": "TTL 300s in production, 60s in staging; revisit if stale-price "
              "complaints exceed 1/week",
})
print("call recorded. Surface B exits.")

call recorded. Surface B exits.


## Surface C — the editor, Tuesday afternoon

A third connection, again starting from nothing but the identifier.

In [6]:
work = open_work(TASK)
print("reopened by identity:", TASK)
print("objective :", work["objective"])
print("calls     :", len(work["calls"]))
for c in work["calls"]:
    print("   answer  :", c["answer"][:64], "...")
    print("   excluded:", c["excluded"])
print()
print("Nothing was summarised. Nothing was retyped.")

reopened by identity: task-ef248d0f
objective : Decide the cache TTL policy for the pricing API
calls     : 1
   answer  : TTL 300s in production, 60s in staging; revisit if stale-price c ...
   excluded: [['e2', 'excluded because budget'], ['e3', 'excluded because budget']]

Nothing was summarised. Nothing was retyped.


## The other path: reconstruction

Now the workaround the chapter takes seriously and rejects. Summarise the work
and hand the summary to the next surface.

In [7]:
def summarise(task_id, max_points=3):
    """A lossy compression of the record - what a summariser plausibly keeps."""
    w = open_work(task_id)
    points = [f"Objective: {w['objective']}"]
    for d in w["decisions"][:max_points - 1]:
        points.append(f"Decided: {d['what']}")
    return "\n".join(points)

summary = summarise(TASK)
print("--- summary handed to the next surface ---")
print(summary)
print("-----------------------------------------")
print()
rejected_terms = ["in-process cache", "rejected", "stale prices"]
print("does the summary mention the rejected approach?",
      any(t in summary for t in rejected_terms))

--- summary handed to the next surface ---
Objective: Decide the cache TTL policy for the pricing API
Decided: TTL is chosen per environment, not globally
Decided: TTL values live in config, not in code
-----------------------------------------

does the summary mention the rejected approach? False


## The prior fills what the context failed to carry

The chapter's mechanism:

> When the selected context fails to carry a project-specific fact, the model
> does not encounter an empty space. Its learned defaults fill it.

The "component" below is a lookup table, not a model. Given a project that
*looks* conventional, it proposes the conventional answer — which is exactly
the approach this project ruled out in 2025.

In [8]:
def conventional_prior(context_text: str) -> str:
    """Stands in for a learned prior. Proposes what such projects usually do."""
    return ("Add a shared in-process cache keyed by request path. "
            "It is the standard approach for this shape of service.")

def propose(context_text: str) -> str:
    """Use the record where it speaks; fall back to convention where it does not."""
    if "in-process cache" in context_text:
        return ("Do NOT use a shared in-process cache keyed by request path: "
                "rejected in 2025 (stale prices across tenants).")
    return conventional_prior(context_text)

record_text = json.dumps(open_work(TASK), indent=1)

print("Given the RECORD:")
print("  ", propose(record_text))
print()
print("Given the SUMMARY:")
print("  ", propose(summary))

Given the RECORD:
   Do NOT use a shared in-process cache keyed by request path: rejected in 2025 (stale prices across tenants).

Given the SUMMARY:
   Add a shared in-process cache keyed by request path. It is the standard approach for this shape of service.


## Observation

In [9]:
from_record  = propose(record_text)
from_summary = propose(summary)

assert "Do NOT" in from_record
assert "Add a shared in-process cache" in from_summary

print("assertion held:")
print("  from the record  -> the exception is respected")
print("  from the summary -> the rejected approach is proposed back")
print()
print("Both answers are fluent. Both are confident. One is the Tuesday editor")
print("from the start of the chapter, proposing what you ruled out on Monday.")

assertion held:
  from the record  -> the exception is respected
  from the summary -> the rejected approach is proposed back

Both answers are fluent. Both are confident. One is the Tuesday editor
from the start of the chapter, proposing what you ruled out on Monday.


## Interpretation

Three results, in the chapter's own vocabulary.

1. **Continuation is not reconstruction.** Surfaces B and C opened the same
   work by identity. Nothing was summarised, and nothing stochastic sat between
   the record and the next surface.
2. **Context is selected, not forgotten.** The compiler excluded material for a
   recorded reason and within a recorded budget. Exclusion is evidence;
   *unrecorded* exclusion is a bug you cannot reproduce.
3. **The prior carries the convention; the record must carry the exception.**
   A summary that keeps the decisions and drops the rejected approach produces
   a competent reconstruction of the wrong project. That is why the chapter
   gives rejected approaches, deliberate departures and negative results
   *priority* for explicit recording — not because a model is helpless without
   them, but because it is capable enough to paper over their absence.

What this does not show: synchronisation, offline operation, concurrent
writers, access control, or retention. The chapter names all five as unbuilt,
and a teaching ledger does not build them either.

In [10]:
import shutil
shutil.rmtree(WORKDIR, ignore_errors=True)
print("temporary ledger removed:", not LEDGER.exists())

temporary ledger removed: False


## Try it yourself

1. **Lower the budget to 10.** Which items survive, and does the exclusion
   trace still let you explain the answer afterwards? Now set the budget so low
   that a *required* item does not fit — what should the compiler do? Chapter 15
   argues it must fail loudly rather than trim.
2. **Make the summariser better.** Let it keep four points, then five. At what
   point does it keep the rejected approach — and would you have known where to
   set that threshold in advance?
3. **Add a second rejected approach** and re-run. Does your summariser drop the
   older one? Which of your own projects has a decision that only one person
   remembers?
4. **Delete the ledger file** between surfaces B and C and watch `open_work`
   raise. Compare that with a chat window closing: which failure would you
   rather have?